# Trace AI Agent Decisions with BigQuery Property Graphs

This Colab notebook walks you through building a queryable BigQuery property graph from raw AI agent event logs, using the BigQuery Agent Analytics SDK.

*BigQuery property graphs, BigQuery Conversational Analytics, and the BigQuery Agent Analytics SDK are currently in Preview on Google Cloud. The BigQuery Agent Analytics Plugin is Generally Available (GA). This notebook uses synthetic data.*

## What you will do

1. **Configure** your GCP project, region, and a BigQuery dataset.
2. **Install** the SDK and authenticate.
3. **Download** the codelab's bundled graph artifacts (DDL, ontology, binding, event generator).
4. **Apply** the property-graph schema.
5. **Generate** synthetic agent events.
6. **Materialize** the decision graph by running `bqaa-materialize-window`.
7. **Query** the graph in GQL to trace any decision end-to-end.
8. **Replay** a past time window with backfill, separately from the regular refresh (advanced).
9. **Clean up** when you are done.

**Total time: about 30 minutes.**

## Where you can run this notebook

* **Vertex AI Colab Enterprise** (recommended for production-aligned GCP work). Open in your project from the [Colab Enterprise console](https://console.cloud.google.com/vertex-ai/colab/notebooks). Imported notebooks auto-authenticate against your project's service account.
* **colab.research.google.com** (free, fastest to try). You will be prompted to authenticate against your GCP project in the setup cell.
* **Local Jupyter** if you have `bigquery-agent-analytics` and `google-cloud-bigquery` installed and `gcloud` authenticated.

Companion codelab: [`docs/codelabs/periodic_materialization.md`](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/blob/main/docs/codelabs/periodic_materialization.md). Companion blog post: [Trace AI Agent Decisions with BigQuery Property Graphs](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/blob/main/docs/blog/periodic_materialization.md).

## 1. Configure your project

Set the three variables for your environment. Everything else in this notebook derives from these.

In [ ]:
import os

# CHANGE THESE THREE LINES to match your GCP environment.
PROJECT_ID = "your-project-id"   # e.g. "my-bqaa-demo"
REGION     = "us-central1"
DATASET    = "agent_analytics_demo"

# Export to the shell environment so `!gcloud ...` and `!bq ...`
# cells pick them up, and so `envsubst < file.yaml` works below.
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"]     = REGION
os.environ["DATASET"]    = DATASET

print(f"PROJECT_ID = {PROJECT_ID}")
print(f"REGION     = {REGION}")
print(f"DATASET    = {DATASET}")

## 2. Authenticate and select your project

**On Vertex AI Colab Enterprise**, you are typically already authenticated as the service account for your project, and `gcloud config set project` is the only thing you need to run.

**On colab.research.google.com**, run the auth cell. It prompts you to sign in to a Google account that has Editor or Owner access on `PROJECT_ID`.

In [ ]:
# Authenticate (skip on Vertex AI Colab Enterprise — already auth'd).
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via google.colab.auth")
except ImportError:
    # Not running in colab.research.google.com — assume gcloud ADC is set
    # (Vertex AI Colab Enterprise, or local Jupyter).
    print("google.colab not available — assuming gcloud Application Default Credentials are already configured.")

!gcloud config set project $PROJECT_ID

## 3. Enable the required APIs

Two GCP APIs are needed: BigQuery and Vertex AI. The Vertex AI API is required because the SDK's default extraction path calls BigQuery's `AI.GENERATE` function. If you later switch to `--extraction-mode=compiled-only`, you can remove this dependency.

In [ ]:
!gcloud services enable \
    bigquery.googleapis.com \
    aiplatform.googleapis.com \
    --project="$PROJECT_ID"

## 4. Create the BigQuery dataset

This codelab uses a **single dataset** for both raw `agent_events` and the materialized graph tables. Production deployments often split events and graph into separate datasets so IAM can be granted narrowly per dataset; for the codelab one dataset keeps things simple.

In [ ]:
!bq --location=US mk --dataset "$PROJECT_ID:$DATASET" || echo "(dataset may already exist — that is fine)"

## 5. Install the SDK

Install the BigQuery Agent Analytics SDK from PyPI. The `bqaa-materialize-window` CLI entry point ships with the package.

In [ ]:
!pip install --quiet --upgrade bigquery-agent-analytics google-cloud-bigquery
!bqaa-materialize-window --help | head -8

## 6. Download the bundled codelab artifacts

The codelab ships five ready-to-use files describing a generic agent decision flow. You do not author any of them yourself; the [README in the artifacts folder](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/blob/main/examples/codelab/periodic_materialization/README.md) explains how to adapt them for your own decision domain.

In [ ]:
# Source of the bundled codelab artifacts.
#
# By default, BASE points at the latest `main` branch — this is the
# right value AFTER PR #243 (which adds this notebook + its bundled
# artifacts) has merged.
#
# If you are running this notebook BEFORE PR #243 has merged, the
# /main/ URL will 404 because the artifacts are not on main yet.
# Override BASE to the PR branch in that case:
#
#   BASE="https://raw.githubusercontent.com/caohy1988/BQAA-SDK-fork/docs/codelab-refresh-pr-240/examples/codelab/periodic_materialization"
#
import os
os.environ["BASE"] = os.environ.get(
    "BASE",
    "https://raw.githubusercontent.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/main/examples/codelab/periodic_materialization",
)
print(f"Downloading codelab artifacts from: {os.environ['BASE']}")

!mkdir -p ~/bqaa-codelab && cd ~/bqaa-codelab && \
  for f in property_graph.sql table_ddl.sql ontology.yaml binding.yaml seed_events.py; do \
    curl -fsSL "$BASE/$f" -o "$f"; \
  done && \
  ls -la

The decision flow these artifacts describe has three node types and two heterogeneous edges:

```
DecisionRequest --[evaluatesOption]--> DecisionOption
              \--[resultedIn]--------> DecisionOutcome
```

* `DecisionRequest` is the question the agent received.
* `DecisionOption` is one alternative the agent considered.
* `DecisionOutcome` records the committed choice and the rationale.

## 7. Apply the property-graph schema

Apply the table DDL first (the property graph references those tables; BigQuery rejects a `CREATE PROPERTY GRAPH` that points at tables that do not yet exist), then the property-graph DDL.

In [ ]:
!cd ~/bqaa-codelab && envsubst < table_ddl.sql      | bq query --use_legacy_sql=false
!cd ~/bqaa-codelab && envsubst < property_graph.sql | bq query --use_legacy_sql=false

You should see five `CREATE TABLE` results and one `CREATE PROPERTY GRAPH` result. The DDL is idempotent — re-running is safe.

### Render the binding placeholders

The materializer reads `binding.yaml` directly; substitute the shell variables once before the materializer reads it.

In [ ]:
!cd ~/bqaa-codelab && envsubst < binding.yaml > binding.rendered.yaml && head -20 binding.rendered.yaml

## 8. Generate sample agent events

In production, the BigQuery Agent Analytics Plugin captures events automatically as your ADK agent runs:

```python
from google.adk.plugins import BigQueryAgentAnalyticsPlugin

plugin = BigQueryAgentAnalyticsPlugin(
    project_id="your-project-id",
    dataset_id="agent_analytics_demo",
)
runner = Runner(agent=root_agent, plugins=[plugin])
```

For this notebook, use the bundled synthetic event generator. It writes the same shape of rows directly to `agent_events`.

In [ ]:
!cd ~/bqaa-codelab && python seed_events.py \
    --project-id "$PROJECT_ID" \
    --dataset-id "$DATASET" \
    --sessions 5

In [ ]:
# Verify the events landed.
!bq query --use_legacy_sql=false \
    "SELECT event_type, COUNT(*) AS n FROM \`$PROJECT_ID.$DATASET.agent_events\` GROUP BY event_type ORDER BY n DESC"

You should see 25 `TOOL_COMPLETED` rows and 5 `AGENT_COMPLETED` rows (each session emits one `submit_request`, three `evaluate_option`, one `commit_outcome`, and one closing `AGENT_COMPLETED` — five tool events plus one agent terminator per session). The `AGENT_COMPLETED` rows are the session terminators the materializer keys on.

## 9. Materialize the decision graph

Run the materializer. With `--lookback-hours 24`, it scans the last day of events, picks out completed sessions, extracts the decision flow from each via `AI.GENERATE`, and writes the corresponding rows into the graph tables.

In [ ]:
!cd ~/bqaa-codelab && bqaa-materialize-window \
    --project-id "$PROJECT_ID" \
    --dataset-id "$DATASET" \
    --ontology ontology.yaml \
    --binding binding.rendered.yaml \
    --lookback-hours 24 \
    --format json

You should see a JSON report with `"ok": true` and `"sessions_materialized": 5`.

**If you see `"ok": false` with `error_code = "empty_extraction"`,** the most common cause is that the Vertex AI API has not propagated yet, or your account is missing `roles/aiplatform.user`. Wait a minute and retry, or grant the role:

```bash
USER_EMAIL=$(gcloud auth list --filter=status:ACTIVE --format="value(account)")
gcloud projects add-iam-policy-binding "$PROJECT_ID" \
    --member="user:$USER_EMAIL" --role="roles/aiplatform.user"
```

In [ ]:
# Verify the graph has rows.
!bq query --use_legacy_sql=false \
    "SELECT COUNT(*) AS decision_requests FROM \`$PROJECT_ID.$DATASET.decision_request\`"

### Two ways to extract decisions from events

The materializer offers two extraction paths. Pick the one that matches your workload:

* **Default extraction.** The easiest path. Uses BigQuery's `AI.GENERATE` to read event content and infer entities and relationships. Works against any event shape with no extra code. This is what the notebook uses.
* **Deterministic extraction** (`--extraction-mode=compiled-only`). The lower-cost, audit-friendly path. Uses a small Python reference extractor you write once for your ontology. No Vertex AI calls, no per-token charges, fully reproducible output. Production deployments choose this when cost predictability or strict reproducibility matters.

> 💡 **Tip.** Deterministic extraction is also the path for regulated workloads that need to remove the Vertex AI dependency from the runtime service account entirely. See the [production deployment guide](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/tree/main/examples/migration_v5/periodic_materialization) for the IAM details.

## 10. Query the decision trace

With the graph populated, the audit question is a single Graph Query Language (GQL) traversal. For each `DecisionRequest`, the query returns every option the agent considered, the final outcome, and the recorded rationale.

In [ ]:
traversal_sql = f"""
SELECT *
FROM GRAPH_TABLE (
  {DATASET}.agent_decisions_graph
  MATCH
    (req:DecisionRequest) -[eo:evaluatesOption]-> (opt:DecisionOption),
    (req)                 -[ri:resultedIn]->      (out:DecisionOutcome)
  COLUMNS (
    req.request_id   AS request,
    req.request_text AS question,
    opt.option_label AS considered,
    opt.confidence   AS score,
    out.status       AS outcome,
    out.rationale    AS rationale
  )
)
ORDER BY request, score DESC;
"""

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
results_df = client.query(traversal_sql).to_dataframe()
results_df

You should see **fifteen rows** in the resulting DataFrame: three options per request, five requests. Each row shows the request, the option the agent considered, its confidence score, the final outcome, and the rationale. For a single decision's full picture, filter by `request_id` to get the row set an audit team needs: the question that came in, the options that were weighed (with scores), and the rationale that was committed.

### Ask the same question in plain English

Not every audit reader writes GQL. With **BigQuery Conversational Analytics** (Preview), your compliance team can ask the same decision-trace question in natural language and get back a structured answer card — no query syntax, no joins to learn:

> *"Why did the agent commit outcome X on request Y?"*

Register the property graph you built in this notebook as a Conversational Analytics knowledge source and the same data is reachable from a chat-style interface. See the [Conversational Analytics documentation](https://cloud.google.com/bigquery/docs/conversational-analytics) for setup.

## 11. Advanced: Replay a past window

Sometimes you need to re-process a past time window: events arrived during an outage, a schema change requires re-extraction, or an audit team asks about a specific historical period. Backfill mode lets you do that **without disturbing the regular refresh schedule** — the replay runs against a fixed start-and-end window you choose, and its progress is tracked separately from the regular refresh.

In a real recovery you would point the backfill at the window where the missed events actually arrived. For this notebook, run a backfill against an **empty historical window** (eight to nine hours ago, before you seeded any events). The replay finishes immediately because there is nothing to materialize, which lets you see the audit trail it produces without re-touching the rows you already wrote in Section 9.

In [ ]:
import datetime, os
now = datetime.datetime.now(datetime.timezone.utc)
os.environ["FROM"] = (now - datetime.timedelta(hours=9)).strftime("%Y-%m-%dT%H:%M:%SZ")
os.environ["TO"]   = (now - datetime.timedelta(hours=8)).strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"Backfill window FROM = {os.environ['FROM']}")
print(f"Backfill window TO   = {os.environ['TO']}")

!cd ~/bqaa-codelab && bqaa-materialize-window \
    --project-id "$PROJECT_ID" \
    --dataset-id "$DATASET" \
    --ontology ontology.yaml \
    --binding binding.rendered.yaml \
    --lookback-hours 1 \
    --backfill --from "$FROM" --to "$TO" \
    --state-key-suffix codelab_backfill_demo \
    --format json

You should see a JSON report with `"sessions_materialized": 0` because the window you picked doesn't overlap with the events you seeded. Now inspect the audit trail this run wrote into the state table:

In [ ]:
# Inspect the state table. You should see at least two rows: a
# `mode = 'steady'` row from the Section 9 materialization and a
# `mode = 'backfill'` row from this run. The two rows have different
# `state_key` values (the --state-key-suffix you passed changes the
# hash input), so the backfill checkpoint sits in its own namespace
# and the regular schedule's progress marker stays untouched.
!bq query --use_legacy_sql=false \
    "SELECT mode, scan_start, scan_end, sessions_materialized, ok \
     FROM \`$PROJECT_ID.$DATASET._bqaa_materialization_state\` \
     ORDER BY run_started_at DESC LIMIT 5"

You should see at least two rows: one from the Section 9 materialization (`mode = 'steady'`) and one from this backfill (`mode = 'backfill'`). The two are independent — the backfill's progress is tracked separately, so it cannot disrupt the regular refresh.

> 💡 **Tip — how the isolation works (optional detail).** Each materializer run writes a row to the `_bqaa_materialization_state` table keyed by a `state_key` hash. Backfill mode mixes the `--state-key-suffix` you pass into that hash, so the backfill writes to a different `state_key` than the regular schedule. Same table, different rows, separate progress markers. Production operators query this table to confirm a catch-up actually ran.

## 12. Production-grade capabilities

The local run you completed in Section 9 uses default behavior. Real deployments care about cost, reliability, and audit posture — the SDK supports each one out of the box.

**What you get by default:**

* **Every run leaves a clear audit trail.** Structured JSON logs go to Cloud Logging, and a per-run row lands in a state table inside your dataset — useful for alerting, dashboards, and answering "did the refresh actually happen?"
* **Transient failures retry automatically.** The materializer retries a small number of times (default two) before flagging a failure.
* **No double-counting.** Progress only advances on sessions that fully succeeded, so retrying a partially-failed run picks up exactly where it left off.

**What you opt into when you need it:**

* **Lower-cost, deterministic extraction** (`--extraction-mode=compiled-only`). Swaps the LLM-based extractor for a small reference-extractor module you write once. Removes the Vertex AI dependency and the per-token cost. Recommended for steady-state production workloads and any audit that requires reproducibility.
* **Catch stuck sessions** (`--max-session-age-hours`). Flags long-running sessions as orphaned so operators can drain them.
* **Replay a past window** (`--backfill --from / --to`). You exercised this in Section 11.
* **Bound the per-run batch size** (`--max-sessions`).

> 💡 **From "run this once" to "run this every six hours."** The SDK ships a deploy script and a Terraform module that wrap `bqaa-materialize-window` as a Cloud Run Job triggered by Cloud Scheduler, with least-privilege service accounts. See the [periodic-materialization deployment guide](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/tree/main/examples/migration_v5/periodic_materialization).

## 13. Clean up

Tear down the BigQuery dataset to avoid ongoing storage charges. Because this notebook uses a single dataset for everything, one command removes the events, the graph tables, and the state table together.

In [ ]:
# UNCOMMENT TO RUN. This permanently deletes the dataset and all tables in it.
# !bq rm -r -f --dataset "$PROJECT_ID:$DATASET"

## Summary

You have:

* Created a BigQuery dataset and applied a property-graph schema describing an agent decision domain.
* Populated `agent_events` with a synthetic event corpus.
* Run `bqaa-materialize-window` to extract a decision graph from those events.
* Replayed a past time window with backfill mode, separately from the regular refresh schedule.
* Queried the resulting graph in GQL and seen the audit-style answer.

The same pattern applies wherever an agent makes consequential decisions: credit underwriting, prior authorization, marketing budget moves, procurement, customer service, internal IT. To build your own decision graph, copy the codelab artifacts as a starting point and adapt the four declarative files (table DDL, property-graph DDL, ontology, binding) to your domain.

## Further reading

* [BigQuery Agent Analytics SDK repository](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK)
* [Companion codelab in markdown form](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/blob/main/docs/codelabs/periodic_materialization.md)
* [Codelab artifacts and adaptation guide](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/tree/main/examples/codelab/periodic_materialization)
* [Periodic-materialization deployment guide](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/tree/main/examples/migration_v5/periodic_materialization)
* [BigQuery property graphs documentation](https://cloud.google.com/bigquery/docs/reference/standard-sql/graph-intro) (Preview)
* [BigQuery Conversational Analytics documentation](https://cloud.google.com/bigquery/docs/conversational-analytics) (Preview)